In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
class DGMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers):
        super(DGMNet, self).__init__()

        self.input = nn.Linear(input_dim, hidden_dim)
        self.hidden = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.output = nn.Linear(hidden_dim, 1)

    def activation(self, x):
        # Swish / SiLU activation (smooth and good for PDEs)
        return x * torch.sigmoid(x)

    def forward(self, t, x):
        # concatenate time and space
        inp = torch.cat([t, x], dim=1)

        h = self.activation(self.input(inp))
        for layer in self.hidden:
            h = self.activation(layer(h))

        return self.output(h)

class PolicyNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers, output_dim):
        super().__init__()

        self.input = nn.Linear(input_dim, hidden_dim)
        self.hidden = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.output = nn.Linear(hidden_dim, output_dim)

    def activation(self, x):
        return x * torch.sigmoid(x)

    def forward(self, t, x):
        inp = torch.cat([t, x], dim=1)
        h = self.activation(self.input(inp))
        for layer in self.hidden:
            h = self.activation(layer(h))
        return self.output(h)

class DeepGalerkinSolver:
    def __init__(self, H, M, C, D, R, sigma,
                 T,
                 alpha_initial,
                 hidden_dim,
                 n_layers,
                 device):

        self.device = device
        self.T = T

        # Convert matrices to tensors
        self.H = torch.tensor(H, dtype=torch.float32, device=self.device)
        self.M = torch.tensor(M, dtype=torch.float32, device=self.device)
        self.C = torch.tensor(C, dtype=torch.float32, device=self.device)
        self.D = torch.tensor(D, dtype=torch.float32, device=self.device)
        self.R = torch.tensor(R, dtype=torch.float32, device=self.device)
        self.sigma = torch.tensor(sigma, dtype=torch.float32, device=self.device)

        self._validate_positive_definite(self.C, "C")
        self._validate_positive_definite(self.R, "R")
        self._validate_strictly_positive_definite(self.D, "D")

        self.dim = self.H.shape[0]

        self.policy_net = DGMNet(self.dim + 1, hidden_dim, n_layers).to(device)

        if alpha_initial is not None:
            self._initialize_policy(alpha_initial)

        self.net = DGMNet(self.dim + 1, hidden_dim, n_layers).to(self.device)

        self.optimizer = optim.Adam(self.net.parameters(), lr=1e-3)

    def _initialize_policy(self, alpha_initial):
      # Simple way: set biases of last layer to alpha_initial
      alpha_initial = torch.tensor(alpha_initial, dtype=torch.float32, device=self.device)
      if self.policy_net.output.bias.shape[0] == 1:
          # If output is scalar, take mean
          self.policy_net.output.bias.data.fill_(alpha_initial.mean())
      else:
          # Otherwise, broadcast
          self.policy_net.output.bias.data[:alpha_initial.shape[0]] = alpha_initial

    def _validate_positive_definite(self, A, name):
        """
        Checks if A is symmetric positive definite.
        Raises ValueError if not.
        """

        # Check symmetry
        if not torch.allclose(A, A.T, atol=1e-6):
            raise ValueError(f"Matrix {name} is not symmetric.")

        # Cholesky test
        try:
            torch.linalg.cholesky(A)
        except RuntimeError:
            raise ValueError(f"Matrix {name} is not positive definite.")

    def _validate_strictly_positive_definite(self, A, name):
        """
        Checks if A is strictly positive definite
        (all eigenvalues strictly > 0).
        """

        # Check symmetry
        if not torch.allclose(A, A.T, atol=1e-6):
            raise ValueError(f"Matrix {name} is not symmetric.")

        # Eigenvalue check
        eigenvalues = torch.linalg.eigvalsh(A)
        min_eig = torch.min(eigenvalues)

        if min_eig <= 0:
            raise ValueError(
                f"Matrix {name} is not strictly positive definite. "
                f"Minimum eigenvalue = {min_eig.item():.6e}"
            )

    def pde_residual(self, t, x):
        t.requires_grad_(True)
        x.requires_grad_(True)

        u = self.net(t, x)

        # Time derivative
        u_t = torch.autograd.grad(
            u, t,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        # First spatial gradient
        grad_u = torch.autograd.grad(
            u, x,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        # Hessian (loop over dimensions)
        hessian = []
        for i in range(self.dim):
            grad_i = torch.autograd.grad(
                grad_u[:, i],
                x,
                grad_outputs=torch.ones_like(grad_u[:, i]),
                create_graph=True
            )[0]
            hessian.append(grad_i)

        hessian = torch.stack(hessian, dim=2)  # shape: (batch, dim, dim)

        # Diffusion term: 1/2 tr(σσ^T Hessian)
        sigma_sigmaT = self.sigma @ self.sigma.T
        diffusion = 0.5 * torch.einsum("ij,bji->b", sigma_sigmaT, hessian)

        # Drift gradient terms
        Hx = x @ self.H.T
        a = self.policy_net(t,x)
        Malpha = a @ self.M.T

        drift_term = (grad_u * (Hx + Malpha)).sum(dim=1)

        # Quadratic source terms
        quad_x = torch.einsum("bi,ij,bj->b", x, self.C, x)
        quad_alpha = torch.einsum("bi,ij,bj->b", a, self.D, a)

        residual = (
            u_t.squeeze()
            + diffusion
            + drift_term
            + quad_x
            + quad_alpha
        )

        return residual

    def loss(self, batch_size):

        # Sample interior points
        t = torch.rand(batch_size, 1, device=self.device) * self.T
        x = torch.randn(batch_size, self.dim, device=self.device)

        res = self.pde_residual(t, x)
        pde_loss = torch.mean(res**2)

        # Terminal condition
        x_T = torch.randn(batch_size, self.dim, device=self.device)
        t_T = torch.ones(batch_size, 1, device=self.device) * self.T

        u_T = self.net(t_T, x_T)
        target = torch.einsum("bi,ij,bj->b", x_T, self.R, x_T)

        terminal_loss = torch.mean((u_T.squeeze() - target)**2)

        return pde_loss + terminal_loss


    def train(self, epochs=5000, batch_size=256):
      """
      Train the model and record loss history and error against a given Monte Carlo mean.

      Parameters:
          epochs : number of training iterations
          batch_size : batch size for loss evaluation
      Returns:
          loss_history : list of training losses
      """

      self.loss_history = []

      for epoch in range(epochs):
          self.optimizer.zero_grad()

          # Compute loss on batch
          loss = self.loss(batch_size)
          loss.backward()

          self.optimizer.step()

          # Record loss
          self.loss_history.append(loss.item())

          if epoch % 250 == 249:
              print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

      return self.loss_history

def compute_hamiltonian_loss(theta_val_net, theta_act_net, H, M, C, D, batch_size=256, device="cpu"):
    """
    Compute the mean Hamiltonian for a batch of samples.

    Parameters
    ----------
    theta_val_net : nn.Module
         Value function network (fixed parameters).
    theta_act_net : nn.Module
        Policy network (parameters to optimise).
    H, M, C, D : torch.Tensor
        Problem matrices (dim x dim).
    batch_size : int
        Number of samples for estimating Hamiltonian.
    device : str
         'cpu' or 'cuda'.

    Returns
    -------
    torch.Tensor
        Mean Hamiltonian over the batch.
    """
    dim = H.shape[0]
    H = H.to(device)
    M = M.to(device)
    C = C.to(device)
    D = D.to(device)

    # Sample batch points
    t = torch.rand((int(batch_size), 1), device=device)
    x = torch.randn((int(batch_size), dim), device=device)
    x.requires_grad_(True)

    # Value function gradient
    v = theta_val_net(t, x)
    grad_v = torch.autograd.grad(
        v, x, grad_outputs=torch.ones_like(v), create_graph=True
    )[0]  # shape (batch, dim)

    # Current policy
    a = theta_act_net(t, x)  # shape (batch, dim)

    # Hamiltonian: H_i = grad_v^T H x + grad_v^T M a + x^T C x + a^T D a
    ham = (grad_v @ H.T) * x  # grad_v^T H x, shape (batch, dim)
    ham = ham.sum(dim=1)  # sum over dimensions
    ham += (grad_v @ M.T * a).sum(dim=1)  # grad_v^T M a
    ham += torch.einsum("bi,ij,bj->b", x, C, x)  # x^T C x
    ham += torch.einsum("bi,ij,bj->b", a, D, a)  # a^T D a

    return ham.mean()

def policy_improvement(theta_val_net, theta_act_net, H, M, C, D, batch_size=64, device="cpu", lr=1e-3, n_steps=200):
      """
      Perform policy improvement: update theta_act to minimize Hamiltonian.

      Parameters
      ----------
      theta_val_net : nn.Module
          Value function network (fixed parameters).
      theta_act_net : nn.Module
          Policy network to update.
      H, M, C, D : torch.Tensor
          Problem matrices.
      batch_size : int
          Number of points to sample per step.
      device : str
          'cpu' or 'cuda'.
      lr : float
          Learning rate.
      n_steps : int
          Gradient steps to take.

      Returns
      -------
      theta_act_net : nn.Module
          Updated policy network.
      """
      optimizer = torch.optim.Adam(theta_act_net.parameters(), lr=lr)
      theta_val_net.eval()  # freeze value function

      for _ in range(n_steps):
          optimizer.zero_grad()
          loss = compute_hamiltonian_loss(theta_val_net, theta_act_net, H, M, C, D, batch_size, device)
          loss.backward()
          optimizer.step()

      return theta_act_net

if __name__ == "__main__":
  n_iter = 5
  alpha_initial = np.array([1,1])
  H = np.array([[0.5, 0.1], [0.1, 0.3]])
  M = np.array([[1.0, 0.5], [0.0, 1.0]])
  C = np.array([[2.0, 0.5], [0.5, 1.0]])
  D = np.array([[2.0, 0.0], [0.0, 1.0]])
  R = np.array([[1.0, 0.0], [0.0, 2.0]])
  sigma = np.array([[0.3, 0.1], [0.0, 0.2]])

  hidden_dim = 64
  n_layers = 4
  epochs = 5000
  T = 1.0

  device = "cuda" if torch.cuda.is_available() else "cpu"

  def policy_iteration(H, M, C, D, R, sigma, T, alpha_initial, hidden_dim, n_layers, epochs, device, n_iter):
    t_test = torch.tensor([[0.0], [0.5], [1.0]], dtype=torch.float32).to(device)   # (3,1)

    x_test = torch.tensor([
      [1.0, 1.0],
      [0.5, -0.5],
      [0.0, 0.0]
    ], dtype=torch.float32).to(device)  # (3,2)

    # initialise policy network
    policy_net = PolicyNet(input_dim=3, hidden_dim=hidden_dim, n_layers=n_layers, output_dim=2).to(device)

    for k in range(n_iter):

        print(f"\n--- Policy Iteration {k} ---")
        # 1. POLICY EVALUATION
        if k==0:
          solver = DeepGalerkinSolver(
              H, M, C, D, R, sigma,
              T=T,
              alpha_initial=alpha_initial,
              hidden_dim=hidden_dim,
              n_layers=n_layers,
              device=device
          )
        else:
            solver = DeepGalerkinSolver(
              H, M, C, D, R, sigma,
              T=T,
              alpha_initial=None,
              hidden_dim=hidden_dim,
              n_layers=n_layers,
              device=device
          )

        solver.policy_net = policy_net  # inject current policy

        solver.train(epochs)

        # 2. POLICY IMPROVEMENT
        policy_net = policy_improvement(
            theta_val_net=solver.net,   # value function
            theta_act_net=policy_net,
            H=solver.H,
            M=solver.M,
            C=solver.C,
            D=solver.D,
            batch_size=256,
            device=device,
        )
        with torch.no_grad():
          values = solver.net(t_test, x_test)
          controls = policy_net(t_test, x_test)

        for i in range(t_test.shape[0]):
          t_val = t_test[i].item()
          x_val = x_test[i].cpu().numpy()

          v = values[i].item()
          a = controls[i].cpu().numpy()

          print(f"(t={t_val}, x={x_val}) -> value={v:.4f}, control={a}")

    return solver.net, policy_net

  policy_iteration(H=H,
                         M=M,
                         C=C,
                         D=D,
                         R=R,
                         sigma=sigma,
                         T=T,
                         alpha_initial = alpha_initial,
                         hidden_dim=hidden_dim,
                         n_layers=n_layers,
                         epochs=epochs,
                         device=device,
                         n_iter=n_iter)



In [ ]:
# Code from exercise 1.1 to solve Riccati ODE
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LQRSolver:
    """
    Solves the LQR problem via Riccati ODE.

    We want to minimise
        J(t,x) = E[ int_t^T (X'CX + a'Da) ds + X_T' R X_T ],
    subject to
        dX = (HX + Ma) dt + sigma dW,  X_t = x.

    The value function is
        v(t,x) = x'S(t)x + int_t^T tr(sigma sigma' S(r)) dr
    where S solves the Riccati ODE and the optimal control is
        a(t,x) = -D^{-1} M' S(t) x.
    """

    def __init__(self, H, M, C, D, R, sigma, T):
        self.H = np.array(H, dtype=float)
        self.M = np.array(M, dtype=float)
        self.C = np.array(C, dtype=float)
        self.D = np.array(D, dtype=float)
        self.R = np.array(R, dtype=float)
        self.sigma = np.array(sigma, dtype=float)
        self.T = float(T)

        self.d = self.H.shape[0]  # state dimension (coursework uses d=2)

        # precompute these since they're used a lot
        self.D_inv = np.linalg.inv(self.D)
        self.MD_invMT = self.M @ self.D_inv @ self.M.T

        self.S_interp = None
        self.integral_interp = None

    def _riccati_rhs(self, tau, S_flat):
        """
        RHS of the Riccati ODE in forward time (tau = T - t).

        The ODE in original time is
            S'(t) = -2H'S + S M D^{-1} M' S - C,   S(T) = R.

        Substituting tau = T - t flips the sign so we can use a standard
        forward solver
            dS/dtau = 2H'S - S M D^{-1} M' S + C.
        """
        S = S_flat.reshape(self.d, self.d)
        dS = 2.0 * self.H.T @ S - S @ self.MD_invMT @ S + self.C
        return dS.reshape(-1)

    def solve_riccati(self, time_grid):
        """
        Solve the Riccati ODE on the given time grid and build interpolators
        for S(t) and the integral term.

        Args:
            time_grid: array of times in [0, T] (numpy or torch)

        Returns:
            S_values: array of shape (len(time_grid), d, d)
        """
        if isinstance(time_grid, torch.Tensor):
            time_grid = time_grid.detach().cpu().numpy()
        time_grid = np.array(time_grid, dtype=float).reshape(-1)

        if len(time_grid) < 2:
            raise ValueError("time_grid must contain at least two points")
        if np.any(np.diff(time_grid) <= 0):
            raise ValueError("time_grid must be strictly increasing")

        # solve forward in tau = T - t, starting from S(T) = R
        tau_grid = self.T - time_grid  # decreasing in t -> increasing in tau
        sol = solve_ivp(
            self._riccati_rhs,
            (0.0, self.T),
            self.R.reshape(-1),
            t_eval=tau_grid[::-1],  # need increasing tau values
            method="RK45",
            rtol=1e-8,
            atol=1e-10,
        )

        if not sol.success:
            raise RuntimeError(f"Riccati solve failed: {sol.message}")

        # reverse back so columns match increasing t
        S_values = sol.y[:, ::-1].T.reshape(-1, self.d, self.d)

        # enforce symmetry (numerical ODE + interpolation can introduce tiny asymmetry)
        S_values = 0.5 * (S_values + np.transpose(S_values, (0, 2, 1)))

        # build interpolator for S(t)
        kind_S = "cubic" if len(time_grid) >= 4 else "linear"
        self.S_interp = [
            [
                interp1d(
                    time_grid,
                    S_values[:, i, j],
                    kind=kind_S,
                    bounds_error=False,
                    fill_value="extrapolate",
                )
                for j in range(self.d)
            ]
            for i in range(self.d)
        ]

        # compute integral term: int_t^T tr(sigma sigma' S(r)) dr
        # this is the noise-driven part of the value function (doesn't depend on x)
        ss = self.sigma @ self.sigma.T
        trace_vals = np.array([np.trace(ss @ S) for S in S_values], dtype=float)

        # integrate backwards using trapezoid rule
        integral = np.zeros(len(time_grid), dtype=float)
        for i in range(len(time_grid) - 2, -1, -1):
            dt = time_grid[i + 1] - time_grid[i]
            integral[i] = integral[i + 1] + 0.5 * dt * (trace_vals[i] + trace_vals[i + 1])

        # linear is safer for the integral term (avoids cubic overshoot)
        self.integral_interp = interp1d(
            time_grid,
            integral,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",
        )

        return S_values

    def _get_S(self, t):
        """Evaluate S(t) at one or more times. Accepts numpy arrays or torch tensors."""
        if self.S_interp is None:
            raise RuntimeError("Call solve_riccati first")

        is_torch = isinstance(t, torch.Tensor)
        t_np = t.detach().cpu().numpy() if is_torch else np.asarray(t, dtype=float)

        shape = t_np.shape
        t_flat = t_np.reshape(-1)

        S_out = np.zeros((len(t_flat), self.d, self.d), dtype=float)
        for i in range(self.d):
            for j in range(self.d):
                S_out[:, i, j] = self.S_interp[i][j](t_flat)

        S_out = S_out.reshape(*shape, self.d, self.d)

        if is_torch:
            return torch.as_tensor(S_out, dtype=t.dtype, device=t.device)
        return S_out

    def value_function(self, t_batch, x_batch):
        """
        Compute v(t, x) = x' S(t) x + int_t^T tr(sigma sigma' S(r)) dr.

        Args:
            t_batch: torch tensor, shape (batch_size,)
            x_batch: torch tensor, shape (batch_size, 1, d)

        Returns:
            v: torch tensor, shape (batch_size, 1)
        """
        if self.S_interp is None or self.integral_interp is None:
            raise RuntimeError("Call solve_riccati first")

        S = self._get_S(t_batch)  # (batch, d, d)
        x = x_batch.squeeze(1)    # (batch, d)

        # quadratic form x' S(t) x
        xSx = torch.sum((x.unsqueeze(1) @ S).squeeze(1) * x, dim=1)

        # integral correction (noise term)
        integral_vals_np = self.integral_interp(t_batch.detach().cpu().numpy())
        integral_vals = torch.as_tensor(integral_vals_np, dtype=t_batch.dtype, device=t_batch.device)

        return (xSx + integral_vals).unsqueeze(1)

    def optimal_control(self, t_batch, x_batch):
        """
        Compute the optimal control a(t, x) = -D^{-1} M' S(t) x.

        Args:
            t_batch: torch tensor, shape (batch_size,)
            x_batch: torch tensor, shape (batch_size, 1, d)

        Returns:
            a: torch tensor, shape (batch_size, d)
        """
        if self.S_interp is None:
            raise RuntimeError("Call solve_riccati first")

        S = self._get_S(t_batch)  # (batch, d, d)
        x = x_batch.squeeze(1)    # (batch, d)

        neg_DinvMT = torch.as_tensor(-self.D_inv @ self.M.T, dtype=t_batch.dtype, device=t_batch.device)

        # compute S(t) x then apply -D^{-1} M'
        Sx = (S @ x.unsqueeze(2)).squeeze(2)           # (batch, d)
        a = (neg_DinvMT @ Sx.unsqueeze(2)).squeeze(2)  # (batch, d)

        return a

if __name__ == "__main__":

    print(f"Using device: {DEVICE}")

    # Use same problem setup matrices as above
    solver = LQRSolver(H, M, C, D, R, sigma, T)
    time_grid = np.linspace(0, T, 1000)
    S_vals = solver.solve_riccati(time_grid)

    print(f"S(0)  =\n{S_vals[0]}")
    print(f"S(T)  =\n{S_vals[-1]}")
    print(f"Terminal error ||S(T) - R|| = {np.linalg.norm(S_vals[-1] - R):.2e}")

    # check value function and control at a few points
    t_test = torch.tensor([0.0, 0.5, 1.0], dtype=torch.float32).to(DEVICE)
    x_test = torch.tensor([[[1.0, 1.0]], [[0.5, -0.5]], [[0.0, 0.0]]], dtype=torch.float32).to(DEVICE)

    v = solver.value_function(t_test, x_test)
    a = solver.optimal_control(t_test, x_test)
    for i in range(3):
        print(f"t={t_test[i]:.1f}, x={x_test[i,0].cpu().numpy()}, v={v[i,0]:.4f}, a={a[i].cpu().numpy()}")